In [0]:
from pyspark.sql import functions as F

VOLUME_PATH = "/Volumes/medalhao/default/vcredit_raw/"


metadados_ingestao = {
    "base_atendentes.csv": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_atendentes", # <--- Corrigido
        "tem_cabecalho": False,
        "colunas": ["id_atendente", "nome_atendente", "nivel_atendimento"]
    },
    "base_motivos.csv": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_motivos", # <--- Corrigido
        "tem_cabecalho": False,
        "colunas": ["id_motivo", "nome_motivo", "categoria", "criticidade"]
    },
    "canais.csv": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_canais", # <--- Corrigido
        "tem_cabecalho": False,
        "colunas": ["nome_canal", "status_canal"]
    },
    "chamados.csv": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_chamados", # <--- Corrigido
        "tem_cabecalho": False,
        "colunas": ["id_chamado", "id_cliente", "id_motivo", "id_canal", "resolvido", 
                    "hora_abertura", "hora_inicio", "hora_finalizacao", "tempo_espera", 
                    "tempo_atendimento", "id_atendente"]
    },
    "clientes.csv": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_clientes", # <--- Corrigido
        "tem_cabecalho": False,
        "colunas": ["id_cliente", "nome", "email", "regiao", "idade"]
    },
    "custos.csv": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_custos", # <--- Corrigido
        "tem_cabecalho": False,
        "colunas": ["id_custo", "id_chamado", "custo"]
    },
    "pesquisa_satisfacao.csv": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_pesquisa_satisfacao", # <--- Corrigido
        "tem_cabecalho": False,
        "colunas": ["id_pesquisa", "id_chamado", "nota_atendimento"]
    },
    "Chamados_Hora.CSV": {
        "tabela": "medalhao_credit.bronze_credit.vcredit_chamados_hora", # <--- Corrigido
        "tem_cabecalho": True, 
        "colunas": []
    }
}

sucesso = 0
erros = 0

for arquivo, config in metadados_ingestao.items():
    try:
        caminho_completo = f"{VOLUME_PATH}{arquivo}"
        tabela_destino = config["tabela"]
        
        if config["tem_cabecalho"]:
            df_raw = spark.read.csv(caminho_completo, header=True, inferSchema=True)
        else:
            df_raw = spark.read.csv(caminho_completo, header=False, inferSchema=True)
            if config["colunas"]:
                df_raw = df_raw.toDF(*config["colunas"])
        
        df_bronze = df_raw.withColumn("ingestion_timestamp", F.current_timestamp())
        
        df_bronze.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(tabela_destino)
        
        print(f"✅ [OK] {arquivo} -> {tabela_destino}")
        sucesso += 1
        
    except Exception as e:
        print(f"Falha ao ingerir {arquivo}: {e}")
        erros += 1

print(f"Ingestão finalizada. Sucessos: {sucesso} | Erros: {erros}")

In [0]:
# 1. Criar o Catálogo do Projeto V-Credit
spark.sql("CREATE CATALOG IF NOT EXISTS medalhao_credit")

# 2. Criar os Schemas (Bronze e Silver)
spark.sql("CREATE SCHEMA IF NOT EXISTS medalhao_credit.bronze_credit")
spark.sql("CREATE SCHEMA IF NOT EXISTS medalhao_credit.silver_credit")

print("✅ Infraestrutura (Catálogo e Schemas) criada/verificada com sucesso!")